# 带 Profile Schema 的聊天机器人

## 回顾

我们介绍了 [LangGraph Memory Store](https://reference.langchain.com/python/langgraph/store/?h=basestor#langgraph.store.base.BaseStore) 作为保存和检索长期记忆的方式。

我们构建了一个同时使用`短期（线程内）`和`长期（跨线程）`记忆的简单聊天机器人。

它在用户聊天时["在热路径中"](https://docs.langchain.com/oss/python/concepts/memory#writing-memories)保存长期[语义记忆](https://docs.langchain.com/oss/python/concepts/memory#semantic-memory)（关于用户的事实）。

## 目标

我们的聊天机器人将记忆保存为字符串。实际应用中，我们通常希望记忆具有结构。

例如，记忆可以是[单个、持续更新的 schema](https://docs.langchain.com/oss/python/concepts/memory#profile)。

在我们的例子中，我们希望是一个单一的用户资料。

我们将扩展聊天机器人，将语义记忆保存到单个[用户资料](https://docs.langchain.com/oss/python/concepts/memory#profile)中。

我们还将介绍一个库 [Trustcall](https://github.com/hinthornw/trustcall)，用于用新信息更新该 schema。

In [ ]:
%%capture --no-stderr
%pip install -U langchain_deepseek langgraph trustcall langchain_core

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()

def _set_env(var: str):
    # 检查变量是否已在 OS 环境变量中设置
    env_value = os.environ.get(var)
    if not env_value:
        # 如果未设置，提示用户输入
        env_value = getpass.getpass(f"{var}: ")
    
    # 为当前进程设置环境变量
    os.environ[var] = env_value

_set_env("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain-academy"

## 定义用户资料 schema

Python 有不同类型可用于[结构化数据](https://docs.langchain.com/oss/python/langchain/models#structured-outputs)，如 TypedDict、字典、JSON 和 [Pydantic](https://docs.pydantic.dev/latest/)。

让我们先使用 TypedDict 来定义用户资料 schema。

In [ ]:
from typing import TypedDict, List

class UserProfile(TypedDict):
    """带类型字段的用户资料 schema"""
    user_name: str  # 用户的偏好名称
    interests: List[str]  # 用户的兴趣列表

## 将 schema 保存到存储中

[LangGraph Store](https://reference.langchain.com/python/langgraph/store/?h=basestor#langgraph.store.base.BaseStore) 接受任何 Python 字典作为 `value`。

In [ ]:
# TypedDict 实例
user_profile: UserProfile = {
    "user_name": "Lance",
    "interests": ["biking", "technology", "coffee"]
}
user_profile

我们使用 [put](https://reference.langchain.com/python/langgraph/store/?h=basestor#langgraph.store.base.BaseStore.put) 方法将 TypedDict 保存到存储中。

In [ ]:
import uuid
from langgraph.store.memory import InMemoryStore

# 初始化内存存储
in_memory_store = InMemoryStore()

# 用于保存记忆的命名空间
user_id = "1"
namespace_for_memory = (user_id, "memory")

# 通过键和值将记忆保存到命名空间
key = "user_profile"
value = user_profile
in_memory_store.put(namespace_for_memory, key, value)

我们使用 [search](https://reference.langchain.com/python/langgraph/store/?h=basestor#langgraph.store.base.BaseStore.search) 通过命名空间从存储中检索对象。

In [ ]:
# 搜索
for m in in_memory_store.search(namespace_for_memory):
    print(m.dict())

我们也可以使用 [get](https://reference.langchain.com/python/langgraph/store/?h=basestor#langgraph.store.base.BaseStore.get) 通过命名空间和键来检索特定对象。

In [ ]:
# 通过命名空间和键获取记忆
profile = in_memory_store.get(namespace_for_memory, "user_profile")
profile.value

## 带 profile schema 的聊天机器人

现在我们知道如何为记忆指定 schema 并将其保存到存储中。

那么，我们如何*创建*具有特定 schema 的记忆呢？

在我们的聊天机器人中，我们[希望从用户对话中创建记忆](https://docs.langchain.com/oss/python/concepts/memory#profile)。

这就是[结构化输出](https://docs.langchain.com/oss/python/langchain/models#structured-outputs)概念有用的地方。

LangChain 的[聊天模型](https://docs.langchain.com/oss/python/langchain/models)接口有一个 [`with_structured_output`](https://docs.langchain.com/oss/python/langchain/models#structured-outputs) 方法来强制结构化输出。

当我们需要强制输出符合某个 schema 并为我们解析输出时，这非常有用。

In [ ]:
_set_env("DEEPSEEK_API_KEY")

让我们将创建的 `UserProfile` schema 传递给 `with_structured_output` 方法。

然后我们可以用一组[消息](https://docs.langchain.com/oss/python/langchain/messages)调用聊天模型，并获得符合我们 schema 的结构化输出。

In [ ]:
from pydantic import BaseModel, Field

from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek

# 初始化模型
model = ChatDeepSeek(model="deepseek-chat", temperature=0)

# 将 schema 绑定到模型
model_with_structure = model.with_structured_output(UserProfile)

# 调用模型，产生与 schema 匹配的结构化输出
structured_output = model_with_structure.invoke([HumanMessage("My name is Lance, I like to bike.")])
structured_output

现在，让我们将其用于聊天机器人中。

这只需要对 `write_memory` 函数做微小改动。

我们使用上面定义的 `model_with_structure` 来生成匹配 schema 的资料。

In [ ]:
from IPython.display import Image, display

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.store.base import BaseStore

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.runnables.config import RunnableConfig

# 聊天机器人指令
MODEL_SYSTEM_MESSAGE = """You are a helpful assistant with memory that provides information about the user. 
If you have memory for this user, use it to personalize your responses.
Here is the memory (it may be empty): {memory}"""

# 从聊天历史和现有记忆中创建新记忆
CREATE_MEMORY_INSTRUCTION = """Create or update a user profile memory based on the user's chat history. 
This will be saved for long-term memory. If there is an existing memory, simply update it. 
Here is the existing memory (it may be empty): {memory}"""

def call_model(state: MessagesState, config: RunnableConfig, store: BaseStore):

    """从存储中加载记忆，并用其个性化聊天机器人的回答。"""
    
    # 从配置中获取用户 ID
    user_id = config["configurable"]["user_id"]

    # 从存储中检索记忆
    namespace = ("memory", user_id)
    existing_memory = store.get(namespace, "user_memory")

    # 为系统提示格式化记忆
    if existing_memory and existing_memory.value:
        memory_dict = existing_memory.value
        formatted_memory = (
            f"Name: {memory_dict.get('user_name', 'Unknown')}\n"
            f"Interests: {', '.join(memory_dict.get('interests', []))}"
        )
    else:
        formatted_memory = None

    # 在系统提示中格式化记忆
    system_msg = MODEL_SYSTEM_MESSAGE.format(memory=formatted_memory)

    # 使用记忆和聊天历史进行回答
    response = model.invoke([SystemMessage(content=system_msg)]+state["messages"])

    return {"messages": response}

def write_memory(state: MessagesState, config: RunnableConfig, store: BaseStore):

    """反思聊天历史，并将记忆保存到存储中。"""
    
    # 从配置中获取用户 ID
    user_id = config["configurable"]["user_id"]

    # 从存储中检索现有记忆
    namespace = ("memory", user_id)
    existing_memory = store.get(namespace, "user_memory")

    # 为系统提示格式化记忆
    if existing_memory and existing_memory.value:
        memory_dict = existing_memory.value
        formatted_memory = (
            f"Name: {memory_dict.get('user_name', 'Unknown')}\n"
            f"Interests: {', '.join(memory_dict.get('interests', []))}"
        )
    else:
        formatted_memory = None
        
    # 在指令中格式化现有记忆
    system_msg = CREATE_MEMORY_INSTRUCTION.format(memory=formatted_memory)

    # 调用模型，产生与 schema 匹配的结构化输出
    new_memory = model_with_structure.invoke([SystemMessage(content=system_msg)]+state['messages'])

    # 覆盖现有的用户资料记忆
    key = "user_memory"
    store.put(namespace, key, new_memory)

# 定义图
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_node("write_memory", write_memory)
builder.add_edge(START, "call_model")
builder.add_edge("call_model", "write_memory")
builder.add_edge("write_memory", END)

# 用于长期（跨线程）记忆的存储
across_thread_memory = InMemoryStore()

# 用于短期（线程内）记忆的检查点器
within_thread_memory = MemorySaver()

# 使用检查点器和存储编译图
graph = builder.compile(checkpointer=within_thread_memory, store=across_thread_memory)

# 查看
display(Image(graph.get_graph(xray=1).draw_mermaid_png()))

In [ ]:
# 我们提供线程 ID 用于短期（线程内）记忆
# 我们提供用户 ID 用于长期（跨线程）记忆
config = {"configurable": {"thread_id": "1", "user_id": "1"}}

# 用户输入
input_messages = [HumanMessage(content="Hi, my name is Lance and I like to bike around San Francisco and eat at bakeries.")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

让我们检查存储中的记忆。

我们可以看到记忆是一个与我们 schema 匹配的字典。

In [ ]:
# 用于保存记忆的命名空间
user_id = "1"
namespace = ("memory", user_id)
existing_memory = across_thread_memory.get(namespace, "user_memory")
existing_memory.value

## 什么时候会失败？

[`with_structured_output`](https://docs.langchain.com/oss/python/langchain/models#structured-outputs) 非常有用，但如果我们处理更复杂的 schema 会发生什么呢？

[这里](https://github.com/hinthornw/trustcall?tab=readme-ov-file#complex-schema)是一个更复杂 schema 的示例，我们将在下面进行测试。

这是一个 [Pydantic](https://docs.pydantic.dev/latest/) 模型，描述用户的通信和信任背摔偏好。

In [ ]:
from typing import List, Optional

class OutputFormat(BaseModel):
    preference: str
    sentence_preference_revealed: str

class TelegramPreferences(BaseModel):
    preferred_encoding: Optional[List[OutputFormat]] = None
    favorite_telegram_operators: Optional[List[OutputFormat]] = None
    preferred_telegram_paper: Optional[List[OutputFormat]] = None

class MorseCode(BaseModel):
    preferred_key_type: Optional[List[OutputFormat]] = None
    favorite_morse_abbreviations: Optional[List[OutputFormat]] = None

class Semaphore(BaseModel):
    preferred_flag_color: Optional[List[OutputFormat]] = None
    semaphore_skill_level: Optional[List[OutputFormat]] = None

class TrustFallPreferences(BaseModel):
    preferred_fall_height: Optional[List[OutputFormat]] = None
    trust_level: Optional[List[OutputFormat]] = None
    preferred_catching_technique: Optional[List[OutputFormat]] = None

class CommunicationPreferences(BaseModel):
    telegram: TelegramPreferences
    morse_code: MorseCode
    semaphore: Semaphore

class UserPreferences(BaseModel):
    communication_preferences: CommunicationPreferences
    trust_fall_preferences: TrustFallPreferences

class TelegramAndTrustFallPreferences(BaseModel):
    pertinent_user_preferences: UserPreferences

现在，让我们尝试使用 `with_structured_output` 方法提取这个 schema。

In [ ]:
from pydantic import ValidationError

# 将 schema 绑定到模型
model_with_structure = model.with_structured_output(TelegramAndTrustFallPreferences)

# 对话
conversation = """Operator: How may I assist with your telegram, sir?
Customer: I need to send a message about our trust fall exercise.
Operator: Certainly. Morse code or standard encoding?
Customer: Morse, please. I love using a straight key.
Operator: Excellent. What's your message?
Customer: Tell him I'm ready for a higher fall, and I prefer the diamond formation for catching.
Operator: Done. Shall I use our "Daredevil" paper for this daring message?
Customer: Perfect! Send it by your fastest carrier pigeon.
Operator: It'll be there within the hour, sir."""

# 调用模型
try:
    model_with_structure.invoke(f"""Extract the preferences from the following conversation:
    <convo>
    {conversation}
    </convo>""")
except ValidationError as e:
    print(e)

如果我们天真地提取更复杂的 schema，即使使用像 `deepseek-chat` 这样的高能力模型，也容易失败。

## 使用 Trustcall 创建和更新 profile schema

正如我们所见，处理 schema 可能会很棘手。

复杂的 schema 难以提取。

此外，更新即使是简单的 schema 也可能带来挑战。

考虑我们上面的聊天机器人。

每次我们选择保存新记忆时，我们都会*从头开始*重新生成 profile schema。

这是低效的，如果 schema 包含大量信息，每次重新生成会浪费模型 token。

更糟的是，从头开始重新生成资料时我们可能会丢失信息。

[TrustCall](https://github.com/hinthornw/trustcall) 的动机正是为了解决这些问题！

这是一个由 LangChain 团队的 [Will Fu-Hinthorn](https://github.com/hinthornw) 开发的开源库，用于更新 JSON schema。

它正是在处理记忆工作时由这些挑战驱动的。

让我们首先在此[消息](https://docs.langchain.com/oss/python/langchain/messages)列表上演示 TrustCall 的简单提取用法。

In [ ]:
# 对话
conversation = [HumanMessage(content="Hi, I'm Lance."), 
                AIMessage(content="Nice to meet you, Lance."), 
                HumanMessage(content="I really like biking around San Francisco.")]

我们使用 `create_extractor`，将模型以及我们的 schema 作为[工具](https://docs.langchain.com/oss/python/langchain/tools)传入。

使用 TrustCall，我们可以以各种方式提供 schema。

例如，我们可以传入 JSON 对象 / Python 字典或 Pydantic 模型。

在底层，TrustCall 使用[工具调用](https://docs.langchain.com/oss/python/langchain/models#tool-calling)来从输入的[消息](https://docs.langchain.com/oss/python/langchain/messages)列表中产生[结构化输出](https://docs.langchain.com/oss/python/langchain/models#structured-outputs)。

要强制 Trustcall 产生结构化输出，我们可以在 `tool_choice` 参数中包含 schema 名称。

我们可以使用上面的对话来调用提取器。

In [ ]:
from trustcall import create_extractor

# Schema
class UserProfile(BaseModel):
    """带类型字段的用户资料 schema"""
    user_name: str = Field(description="The user's preferred name")
    interests: List[str] = Field(description="A list of the user's interests")

# 初始化模型
model = ChatDeepSeek(model="deepseek-chat", temperature=0)

# 创建提取器
trustcall_extractor = create_extractor(
    model,
    tools=[UserProfile],
    tool_choice="UserProfile"
)

# 指令
system_msg = "Extract the user profile from the following conversation"

# 调用提取器
result = trustcall_extractor.invoke({"messages": [SystemMessage(content=system_msg)]+conversation})

当我们调用提取器时，我们得到以下内容：

* `messages`：包含工具调用的 `AIMessages` 列表。
* `responses`：与我们的 schema 匹配的解析后的工具调用结果。
* `response_metadata`：在更新现有工具调用时适用。它说明哪些响应对应于哪些现有对象。

In [ ]:
for m in result["messages"]: 
    m.pretty_print()

In [ ]:
schema = result["responses"]
schema

In [ ]:
schema[0].model_dump()

In [ ]:
result["response_metadata"]

让我们看看如何使用它来*更新*资料。

对于更新，TrustCall 接收一组消息以及现有的 schema。

核心思想是它提示模型产生 [JSON Patch](https://jsonpatch.com/) 来仅更新 schema 的相关部分。

这比天真地覆盖整个 schema 更不容易出错。

也更高效，因为模型只需要生成发生更改的 schema 部分。

我们可以将现有 schema 保存为字典。

我们可以使用 `model_dump()` 将 Pydantic 模型实例序列化为字典。

我们将其与 schema 名称 `UserProfile` 一起传递给 `"existing"` 参数。

In [ ]:
# 更新对话
updated_conversation = [HumanMessage(content="Hi, I'm Lance."), 
                        AIMessage(content="Nice to meet you, Lance."), 
                        HumanMessage(content="I really like biking around San Francisco."),
                        AIMessage(content="San Francisco is a great city! Where do you go after biking?"),
                        HumanMessage(content="I really like to go to a bakery after biking."),]

# 更新指令
system_msg = f"""Update the memory (JSON doc) to incorporate new information from the following conversation"""

# 使用更新的指令和带对应工具名称（UserProfile）的现有资料调用提取器
result = trustcall_extractor.invoke({"messages": [SystemMessage(content=system_msg)]+updated_conversation}, 
                                    {"existing": {"UserProfile": schema[0].model_dump()}})

In [ ]:
for m in result["messages"]: 
    m.pretty_print()

In [ ]:
result["response_metadata"]

In [ ]:
updated_schema = result["responses"][0]
updated_schema.model_dump()

LangSmith 追踪：

https://smith.langchain.com/public/229eae22-1edb-44c6-93e6-489124a43968/r

现在，让我们也测试 Trustcall 处理我们之前看到的[复杂 schema](https://github.com/hinthornw/trustcall?tab=readme-ov-file#complex-schema)。

In [ ]:
bound = create_extractor(
    model,
    tools=[TelegramAndTrustFallPreferences],
    tool_choice="TelegramAndTrustFallPreferences",
)

# 对话
conversation = """Operator: How may I assist with your telegram, sir?
Customer: I need to send a message about our trust fall exercise.
Operator: Certainly. Morse code or standard encoding?
Customer: Morse, please. I love using a straight key.
Operator: Excellent. What's your message?
Customer: Tell him I'm ready for a higher fall, and I prefer the diamond formation for catching.
Operator: Done. Shall I use our "Daredevil" paper for this daring message?
Customer: Perfect! Send it by your fastest carrier pigeon.
Operator: It'll be there within the hour, sir."""

result = bound.invoke(
    f"""Extract the preferences from the following conversation:
<convo>
{conversation}
</convo>"""
)

# 提取偏好
result["responses"][0]

追踪：

https://smith.langchain.com/public/5cd23009-3e05-4b00-99f0-c66ee3edd06e/r

更多示例请查看概述视频[这里](https://www.youtube.com/watch?v=-H4s0jQi-QY)。

## 带 profile schema 更新的聊天机器人

现在，让我们将 Trustcall 引入聊天机器人，来创建*和更新*记忆资料。

In [ ]:
from IPython.display import Image, display

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.runnables.config import RunnableConfig
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.base import BaseStore

# 初始化模型
model = ChatDeepSeek(model="deepseek-chat", temperature=0)

# Schema
class UserProfile(BaseModel):
    """ 用户资料 """
    user_name: str = Field(description="The user's preferred name")
    user_location: str = Field(description="The user's location")
    interests: list = Field(description="A list of the user's interests")

# 创建提取器
trustcall_extractor = create_extractor(
    model,
    tools=[UserProfile],
    tool_choice="UserProfile", # 强制使用 UserProfile 工具
)

# 聊天机器人指令
MODEL_SYSTEM_MESSAGE = """You are a helpful assistant with memory that provides information about the user. 
If you have memory for this user, use it to personalize your responses.
Here is the memory (it may be empty): {memory}"""

# 提取指令
TRUSTCALL_INSTRUCTION = """Create or update the memory (JSON doc) to incorporate information from the following conversation:"""

def call_model(state: MessagesState, config: RunnableConfig, store: BaseStore):

    """从存储中加载记忆，并用其个性化聊天机器人的回答。"""
    
    # 从配置中获取用户 ID
    user_id = config["configurable"]["user_id"]

    # 从存储中检索记忆
    namespace = ("memory", user_id)
    existing_memory = store.get(namespace, "user_memory")

    # 为系统提示格式化记忆
    if existing_memory and existing_memory.value:
        memory_dict = existing_memory.value
        formatted_memory = (
            f"Name: {memory_dict.get('user_name', 'Unknown')}\n"
            f"Location: {memory_dict.get('user_location', 'Unknown')}\n"
            f"Interests: {', '.join(memory_dict.get('interests', []))}"      
        )
    else:
        formatted_memory = None

    # 在系统提示中格式化记忆
    system_msg = MODEL_SYSTEM_MESSAGE.format(memory=formatted_memory)

    # 使用记忆和聊天历史进行回答
    response = model.invoke([SystemMessage(content=system_msg)]+state["messages"])

    return {"messages": response}

def write_memory(state: MessagesState, config: RunnableConfig, store: BaseStore):

    """反思聊天历史，并将记忆保存到存储中。"""
    
    # 从配置中获取用户 ID
    user_id = config["configurable"]["user_id"]

    # 从存储中检索现有记忆
    namespace = ("memory", user_id)
    existing_memory = store.get(namespace, "user_memory")
        
    # 从列表中获取资料值，并将其转换为 JSON 文档
    existing_profile = {"UserProfile": existing_memory.value} if existing_memory else None
    
    # 调用提取器
    result = trustcall_extractor.invoke({"messages": [SystemMessage(content=TRUSTCALL_INSTRUCTION)]+state["messages"], "existing": existing_profile})
    
    # 将更新后的资料获取为 JSON 对象
    updated_profile = result["responses"][0].model_dump()

    # 保存更新后的资料
    key = "user_memory"
    store.put(namespace, key, updated_profile)

# 定义图
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_node("write_memory", write_memory)
builder.add_edge(START, "call_model")
builder.add_edge("call_model", "write_memory")
builder.add_edge("write_memory", END)

# 用于长期（跨线程）记忆的存储
across_thread_memory = InMemoryStore()

# 用于短期（线程内）记忆的检查点器
within_thread_memory = MemorySaver()

# 使用检查点器和存储编译图
graph = builder.compile(checkpointer=within_thread_memory, store=across_thread_memory)

# 查看
display(Image(graph.get_graph(xray=1).draw_mermaid_png()))

In [ ]:
# 我们提供线程 ID 用于短期（线程内）记忆
# 我们提供用户 ID 用于长期（跨线程）记忆
config = {"configurable": {"thread_id": "1", "user_id": "1"}}

# 用户输入
input_messages = [HumanMessage(content="Hi, my name is Lance")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

In [ ]:
# 用户输入
input_messages = [HumanMessage(content="I like to bike around San Francisco")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

In [ ]:
# 用于保存记忆的命名空间
user_id = "1"
namespace = ("memory", user_id)
existing_memory = across_thread_memory.get(namespace, "user_memory")
existing_memory.dict()

In [ ]:
# 以 JSON 对象保存的用户资料
existing_memory.value

In [ ]:
# 用户输入
input_messages = [HumanMessage(content="I also enjoy going to bakeries")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

在新线程中继续对话。

In [ ]:
# 我们提供线程 ID 用于短期（线程内）记忆
# 我们提供用户 ID 用于长期（跨线程）记忆
config = {"configurable": {"thread_id": "2", "user_id": "1"}}

# 用户输入
input_messages = [HumanMessage(content="What bakeries do you recommend for me?")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

追踪：

https://smith.langchain.com/public/f45bdaf0-6963-4c19-8ec9-f4b7fe0f68ad/r

## Studio

![Screenshot 2024-10-30 at 11.26.31 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/6732d0437060f1754ea79908_Screenshot%202024-11-11%20at%207.48.53%E2%80%AFPM.png)